[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [2]:
import torch
import math

In [7]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B,S,D=Q.shape
    scale=1.0/math.sqrt(D)
    output=torch.zeros_like(Q)

    for b in range(B):
      for i in range(0,S,block_size):
        i_end=min(i+block_size,S)
        q_block=Q[b,i:i_end,:]
        Br=i_end-i

        row_max=torch.full((Br,1),float("-inf"))
        row_sum=torch.zeros((Br,1))
        row_acc=torch.zeros((Br,D))
        for j in range (0,S,block_size):
          j_end=min(j+block_size,S)
          k_block=K[b,j:j_end,:]
          v_block=V[b,j:j_end,:]

          attention_scores=torch.matmul(q_block,k_block.transpose(-1,-2))*scale
          block_max,_=torch.max(attention_scores,dim=-1,keepdim=True)

          new_max=torch.max(row_max,block_max)
          alpha=torch.exp(row_max-new_max)
          beta=torch.exp(attention_scores-new_max)

          row_sum = row_sum*alpha + torch.sum(beta, dim=-1, keepdim=True)

          row_acc=row_acc*alpha+torch.matmul(beta,v_block)

          row_max=new_max

        output[b, i:i_end, :] = row_acc / row_sum
    return output


In [8]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [9]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (17.3ms)
  ✅ [2/4] Non-aligned block size (3.9ms)
  ✅ [3/4] Block size invariant (2.7ms)
  ✅ [4/4] Gradient flow (70.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (94.2ms total)
  Progress saved. Run status() to see your dashboard.

